# Joint Angle Estimation using EKF

### Data initialization

In [ ]:
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

In [2]:
from google.colab import drive
import open_dataset_store as ds
from open_dataset_store.plotting import DataPlotter
import plotly.express as px

print('Open Dataset Store Version :',ds.__version__)

drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"

store = ds.quick_start(dataset_path, backend='local')

Mounted at /content/drive
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)


In [3]:
summary_stats = store.summary()
store.list_entries("experiments")

📊 Dataset Store Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
------------------------------
Entities (Total: 3)
  - subjects: 3
------------------------------
Entries (Total: 7)
  - experiments: 6
  - device: 1


,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files,date,time,weight_kg,notes,related_files
0,experiments_0001,sub_0001,1778229520,Clean & Jerk,raw_data/experiments/experiments_0001_sub_0001...,{},2026-05-08,14:08,20,"5 reps, Sensor detached - left ankle last 3 reps",{'front_video': 'related_files/experiments/exp...
1,experiments_0002,sub_0001,1778230197,Clean & Jerk,raw_data/experiments/experiments_0002_sub_0001...,{},2026-05-08,14:20,20,"5 reps, Sensor detached - left ankle in rep 3",{'raw_zip': 'related_files/experiments/experim...
2,experiments_0003,sub_0002,1778666861,Clean & Jerk,raw_data/experiments/experiments_0003_sub_0002...,{},2026-05-13,15:37,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...
3,experiments_0004,sub_0002,1778667119,Clean & Jerk,raw_data/experiments/experiments_0004_sub_0002...,{},2026-05-13,15:42,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...
4,experiments_0005,sub_0003,1778670186,Clean & Jerk,raw_data/experiments/experiments_0005_sub_0003...,{},2026-05-13,16:33,20,"5 reps, Device 6 (Right Arm and wrist) are mis...",{'raw_zip': 'related_files/experiments/experim...
5,experiments_0006,sub_0003,1778671087,Clean & Jerk,raw_data/experiments/experiments_0006_sub_0003...,{},2026-05-13,16:48,20,"5 reps, Complete",{'raw_zip': 'related_files/experiments/experim...


### Data Cleanup

In [ ]:
df = store.get_entry_raw_data("experiments", "experiments_0001")

In [9]:
print(list(df))

['Timestamp', 'device1_IMU1_AX', 'device1_IMU1_AY', 'device1_IMU1_AZ', 'device1_IMU1_GX', 'device1_IMU1_GY', 'device1_IMU1_GZ', 'device1_IMU2_AX', 'device1_IMU2_AY', 'device1_IMU2_AZ', 'device1_IMU2_GX', 'device1_IMU2_GY', 'device1_IMU2_GZ', 'device2_IMU1_AX', 'device2_IMU1_AY', 'device2_IMU1_AZ', 'device2_IMU1_GX', 'device2_IMU1_GY', 'device2_IMU1_GZ', 'device2_IMU2_AX', 'device2_IMU2_AY', 'device2_IMU2_AZ', 'device2_IMU2_GX', 'device2_IMU2_GY', 'device2_IMU2_GZ', 'device3_IMU1_AX', 'device3_IMU1_AY', 'device3_IMU1_AZ', 'device3_IMU1_GX', 'device3_IMU1_GY', 'device3_IMU1_GZ', 'device3_IMU2_AX', 'device3_IMU2_AY', 'device3_IMU2_AZ', 'device3_IMU2_GX', 'device3_IMU2_GY', 'device3_IMU2_GZ', 'device4_IMU1_AX', 'device4_IMU1_AY', 'device4_IMU1_AZ', 'device4_IMU1_GX', 'device4_IMU1_GY', 'device4_IMU1_GZ', 'device4_IMU2_AX', 'device4_IMU2_AY', 'device4_IMU2_AZ', 'device4_IMU2_GX', 'device4_IMU2_GY', 'device4_IMU2_GZ', 'device5_IMU1_AX', 'device5_IMU1_AY', 'device5_IMU1_AZ', 'device5_IMU1_GX'

In [10]:
summary = store.get_df_summary(df, detailed=True)

--- Data Summary (Detailed) ---


,Column Name,Data Type,Total Records,Missing Values,Min,Max,Mean,Std Dev,Unique Values,Most Frequent
0,Timestamp,int64,7713,0,1.000,7713.000,3857.000000,2226.695646,NaN,NaN
1,device1_IMU1_AX,float64,7713,0,-39.225,22.479,-9.177935,2.837695,NaN,NaN
2,device1_IMU1_AY,float64,7713,0,-39.225,39.227,1.988806,4.021561,NaN,NaN
3,device1_IMU1_AZ,float64,7713,0,-27.088,29.772,0.183931,1.975454,NaN,NaN
4,device1_IMU1_GX,float64,7713,0,-539.116,759.177,-0.097107,36.673109,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
152,device10_IMU3_AY,float64,7713,0,-38.085,33.767,-1.503156,3.467715,NaN,NaN
153,device10_IMU3_AZ,float64,7713,0,-39.225,39.227,-1.489783,2.372924,NaN,NaN
154,device10_IMU3_GX,float64,7713,0,-632.287,557.591,-0.037924,43.888257,NaN,NaN
155,device10_IMU3_GY,float64,7713,0,-578.201,703.750,0.326391,36.883868,NaN,NaN


In [11]:
fig = px.line(
    df, 
    x='Timestamp', 
    y=['device1_IMU1_AX','device1_IMU1_AY','device1_IMU1_AZ'], 
    title='Device 1 IMU Accelerometer Data Over Time'
)

fig.show()

## Link Orientation Estimation

In [28]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
import plotly.express as px
import plotly.graph_objects as go



def hat(v):
    """Maps a 3D vector to a skew-symmetric matrix in so(3)."""
    return np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def exp_so3(omega):
    """Matrix exponential from so(3) to SO(3)."""
    theta = np.linalg.norm(omega)
    if theta < 1e-10:
        return np.eye(3)
    K = hat(omega / theta)
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)

# =====================================================================
# 1. Configuration & A-Pose Definitions
# =====================================================================

# For independent EKFs, we process one IMU per bone. 
# If you have multiple IMUs per bone, you can either average their raw data first, 
# or just select the primary one.
link_mapping = {
    'Upper_Arm_Right': 'device2_IMU1',
    'Lower_Arm_Right': 'device6_IMU1',
    'Upper_Leg_Right': 'device1_IMU1',
    'Lower_Leg_Right': 'device10_IMU1'
}

def get_apose_orientations():
    """
    Defines the global orientation of each body segment (Bone) when the subject 
    stands in a standard A-Pose facing Global +X.
    """
    alignments = {}
    
    # Legs: Pointing straight down (Z = -1)
    leg_rot = R.from_euler('y', 180, degrees=True).as_matrix()
    alignments['Upper_Leg_Right'] = leg_rot
    alignments['Lower_Leg_Right'] = leg_rot
    
    # Arms: A-Pose (arms down, angled outward by ~20 degrees)
    r_arm_rot = R.from_euler('yx', [180, -20], degrees=True).as_matrix()
    alignments['Upper_Arm_Right'] = r_arm_rot
    alignments['Lower_Arm_Right'] = r_arm_rot
    
    return alignments

# =====================================================================
# 2. Automated Steady-State Detection
# =====================================================================

def find_steady_state(df, imu_cols, window=50):
    """Finds the continuous block of data with minimum variance (A-Pose)."""
    df_combined_var = pd.Series(0.0, index=df.index)
    
    for imu in imu_cols:
        acc_cols = [f'{imu}_AX', f'{imu}_AY', f'{imu}_AZ']
        gyr_cols = [f'{imu}_GX', f'{imu}_GY', f'{imu}_GZ']
        
        acc_norm = np.linalg.norm(df[acc_cols].values, axis=1)
        gyr_norm = np.linalg.norm(df[gyr_cols].values, axis=1)
        
        df_combined_var += pd.Series(acc_norm, index=df.index).rolling(window=window).var().fillna(np.inf)
        df_combined_var += pd.Series(gyr_norm, index=df.index).rolling(window=window).var().fillna(np.inf)
        
    end_idx = df_combined_var.idxmin()
    start_idx = max(0, end_idx - window + 1)
    
    steady_means = {}
    for imu in imu_cols:
        acc_cols = [f'{imu}_AX', f'{imu}_AY', f'{imu}_AZ']
        gyr_cols = [f'{imu}_GX', f'{imu}_GY', f'{imu}_GZ']
        steady_means[imu] = {
            'acc_mean': df.loc[start_idx:end_idx, acc_cols].mean().values,
            'gyr_mean': df.loc[start_idx:end_idx, gyr_cols].mean().values
        }
    return steady_means, start_idx, end_idx

# =====================================================================
# 3. Single-IMU Intrinsic EKF Step Function
# =====================================================================

def single_imu_iekf_step(R_k, b_k, P_k, acc_meas, gyr_meas, dt, 
                         Q_static, Q_bias, R_meas_noise, alpha_Q=0.01, Q_threshold=0.5):
    """
    Performs one step of a completely independent IEKF for a single sensor.
    State is 6D: [Rotation Error (3), Gyro Bias Error (3)]
    """
    state_dim = 6
    
    # --- Prediction Step ---
    omega_corrected = gyr_meas - b_k
    R_kp1_k = R_k @ exp_so3(omega_corrected * dt)
    b_kp1_k = np.copy(b_k)
    
    # Dynamic Process Noise
    omega_norm = np.linalg.norm(omega_corrected)
    Q_R_dynamic = Q_static + alpha_Q * max(0, omega_norm - Q_threshold) * np.eye(3)
    
    Q = np.zeros((6, 6))
    Q[0:3, 0:3] = Q_R_dynamic
    Q[3:6, 3:6] = Q_bias
        
    # State Transition Matrix Phi
    Phi = np.eye(6)
    Phi[0:3, 3:6] = -dt * np.eye(3) # Simplified adjoint for bias propagation
        
    P_kp1_k = Phi @ P_k @ Phi.T + Q
    
    # --- Measurement Update (Gravity Only) ---
    gravity = np.array([0, 0, 1.0])
    acc_norm = np.linalg.norm(acc_meas)
    
    if acc_norm > 1e-5:
        acc_meas = acc_meas / acc_norm
        
    z_hat = R_kp1_k.T @ gravity
    H = np.zeros((3, 6))
    H[0:3, 0:3] = hat(z_hat)
    
    # Motion Artifact Rejection
    g_local = 9.81
    motion_penalty = (acc_norm - g_local)**2
    R_mat = np.eye(3) * (R_meas_noise * (1 + 10.0 * motion_penalty))
    
    # Kalman Gain & Update
    innov = acc_meas - z_hat
    S = H @ P_kp1_k @ H.T + R_mat
    K = P_kp1_k @ H.T @ np.linalg.inv(S)
    
    delta_x = K @ innov
    R_kp1_kp1 = R_kp1_k @ exp_so3(delta_x[0:3])
    b_kp1_kp1 = b_kp1_k + delta_x[3:6]
    
    I_KH = np.eye(6) - K @ H
    P_kp1_kp1 = I_KH @ P_kp1_k @ I_KH.T + K @ R_mat @ K.T
    
    return R_kp1_kp1, b_kp1_kp1, P_kp1_kp1

def initialize_arbitrary_rotation(acc_mean):
    """Initializes the EKF with a mathematically valid, but yaw-arbitrary, rotation."""
    Z = acc_mean / np.linalg.norm(acc_mean)
    # Pick ANY orthogonal axis (yaw doesn't matter, it gets calibrated out)
    ref = np.array([0, 1.0, 0])
    X = np.cross(ref, Z)
    if np.linalg.norm(X) < 1e-5:
        X = np.cross(np.array([1.0, 0, 0]), Z)
    X = X / np.linalg.norm(X)
    Y = np.cross(Z, X)
    return np.column_stack((X, Y, Z))

# =====================================================================
# 4. Main Processing & Post-Calibration Pipeline
# =====================================================================

def process_dataset(df, dt=0.01):
    """Runs independent EKFs, then applies Anatomical Calibration."""
    
    # 1. Find the A-Pose time window
    all_imus = list(link_mapping.values())
    steady_means, start_idx, end_idx = find_steady_state(df, all_imus)
    print(f"A-Pose detected at indices {start_idx} to {end_idx}")
    
    # 2. Get Theoretical Bone Orientations during A-Pose
    apose_B_G = get_apose_orientations()
    
    # Dictionaries to store results
    sensor_orientations = {link: [] for link in link_mapping.keys()}
    bone_orientations = {link: [] for link in link_mapping.keys()}
    
    # EKF Tuning Parameters
    Q_static = 1e-4 * np.eye(3)
    Q_bias = 1e-7 * np.eye(3)
    R_meas_noise = 0.05
    
    # 3. Process Each Link Independently
    adapters_S_to_B = {}
    
    for link, imu in link_mapping.items():
        if f'{imu}_AX' not in df.columns:
            continue
            
        # Initialize EKF arbitrarily
        R_k = initialize_arbitrary_rotation(steady_means[imu]['acc_mean'])
        b_k = steady_means[imu]['gyr_mean']
        P_k = np.eye(6) * 0.1
        
        # Calculate Adapter (Sensor to Bone) based on the first initialized frame
        # R_B^G = R_S^G * R_S^B  =>  R_S^B = (R_S^G)^T * R_B^G
        R_S_G_initial = R_k
        R_B_G_apose = apose_B_G[link]
        adapters_S_to_B[link] = R_S_G_initial.T @ R_B_G_apose
        
        link_rotations_bone = []
        
        # Time Loop
        for index, row in df.iterrows():
            acc_meas = row[[f'{imu}_AX', f'{imu}_AY', f'{imu}_AZ']].values.astype(float)
            gyr_meas = row[[f'{imu}_GX', f'{imu}_GY', f'{imu}_GZ']].values.astype(float) * (np.pi / 180.0)
            
            # Step the Independent EKF
            R_k, b_k, P_k = single_imu_iekf_step(
                R_k, b_k, P_k, acc_meas, gyr_meas, dt, 
                Q_static, Q_bias, R_meas_noise
            )
            
            # Post-Calibration: Convert plastic casing rotation to true bone rotation
            R_Bone_Global = R_k @ adapters_S_to_B[link]
            
            link_rotations_bone.append(R_Bone_Global)
            
        bone_orientations[link] = link_rotations_bone
        print(f"Processed {link} (IMU: {imu})")
        
    return bone_orientations

# =====================================================================
# 5. Extract Joint Angles
# =====================================================================

def compute_joint_angles(bone_orientations):
    """
    Computes anatomical joint angles (e.g. Elbow Flexion) from the calibrated bones.
    """
    joint_angles = {}
    
    # Example: Right Elbow
    if 'Upper_Arm_Right' in bone_orientations and 'Lower_Arm_Right' in bone_orientations:
        upper_arm = bone_orientations['Upper_Arm_Right']
        lower_arm = bone_orientations['Lower_Arm_Right']
        
        elbow_flexion = []
        for R_upper, R_lower in zip(upper_arm, lower_arm):
            # Relative rotation: R_Upper^T * R_Lower
            R_joint = R_upper.T @ R_lower
            
            # Extract Euler angles (depending on your specific anatomical axes)
            # Assuming Z is the hinge axis for the elbow
            euler = R.from_matrix(R_joint).as_euler('xyz', degrees=True)
            elbow_flexion.append(euler)
            
        joint_angles['Right_Elbow'] = elbow_flexion
        
    # Example: Right Knee
    if 'Upper_Leg_Right' in bone_orientations and 'Lower_Leg_Right' in bone_orientations:
        upper_leg = bone_orientations['Upper_Leg_Right']
        lower_leg = bone_orientations['Lower_Leg_Right']
        
        knee_flexion = []
        for R_upper, R_lower in zip(upper_leg, lower_leg):
            R_joint = R_upper.T @ R_lower
            euler = R.from_matrix(R_joint).as_euler('xyz', degrees=True)
            knee_flexion.append(euler)
            
        joint_angles['Right_Knee'] = knee_flexion
        
    return joint_angles


In [29]:

# 1. Process data through Independent EKFs + Calibration adapter
bone_orientations = process_dataset(df, dt=0.01)

# 2. Compute valid Joint Angles
joint_angles = compute_joint_angles(bone_orientations)

# 3. Plotting Example (Right Knee)
if 'Right_Knee' in joint_angles:
    angles_array = np.array(joint_angles['Right_Knee'])
    
    # X, Y, Z depend on your specific standard, usually X or Z is flexion/extension
    plot_df = pd.DataFrame({
        'Frame': np.arange(len(angles_array)),
        'Flexion/Extension': angles_array[:, 0], # Change index based on anatomical axis
        'Abduction/Adduction': angles_array[:, 1],
        'Internal/External Rotation': angles_array[:, 2]
    })
    
    fig = px.line(plot_df, x='Frame', y=['Flexion/Extension', 'Abduction/Adduction', 'Internal/External Rotation'],
                    title="True Anatomical Joint Angles: Right Knee")
    fig.show()


A-Pose detected at indices 155 to 204
Processed Upper_Arm_Right (IMU: device2_IMU1)
Processed Lower_Arm_Right (IMU: device6_IMU1)
Processed Upper_Leg_Right (IMU: device1_IMU1)
Processed Lower_Leg_Right (IMU: device10_IMU1)
